In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import linear_sum_assignment
from IPython.display import display
from matplotlib.ticker import MaxNLocator, MultipleLocator, FormatStrFormatter

# Run from the project root or from this notebook's results folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'margin_dictionary_learning.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook within the project containing margin_dictionary_learning.py.')
sys.path.insert(0, str(ROOT))
# Exact source version recorded in the saved evidence results.
from evidence_results.historical_model import MarginDictionaryLearning

RUN_FITS = False  # False: redraw saved results. True: rerun and replace the CSV.
RESUME_FITS = False  # Only with RUN_FITS=True: continue a compatible checkpoint.
PLOT_SETTINGS_ID = None

SEEDS = tuple(range(1, 51))
CALIBRATION_SEEDS = tuple(range(101, 111))
LAMBDA_CANDIDATES = (0.2, 0.3, 0.4)
LAMBDA_STEP = 0.1  # Extend an improving boundary; retain strictly positive penalties.
N_TRAIN, N_TEST, P, K, SPARSITY, SNR = 1500, 1000, 40, 400, 6, 10.0
N_UPDATES = 5000
N_CALIBRATION_UPDATES = 5000
LEARNING_RATE = 0.01
CODE_TOL = 1e-5
CODE_MAX_ITER = 5000
RECORD_EVERY = 100
CAP_QUANTILE = 0.85
SIGMOID_T = 0.4
SMOOTH_T = 0.05  # Fixed temperature in margin units; never scaled by the cap.
DEVICE = 'auto'
METHODS = ('ordinary', 'clipped', 'active_sigmoid', 'sigmoid', 'smooth_clipped')
LABELS = dict(ordinary='Ordinary', clipped='Clipped', active_sigmoid='Active sigmoid',
              sigmoid='Full sigmoid', smooth_clipped='Smooth clipped')
OUTPUT_DIR = ROOT / 'evidence_results'
CSV_PATH = OUTPUT_DIR / 'evidence_scores.csv'
OUTPUT_DIR.mkdir(exist_ok=True)
torch.set_num_threads(2)
MODEL_SHA256 = hashlib.sha256((ROOT / 'evidence_results' / 'historical_model.py').read_bytes()).hexdigest()
records = (pd.read_csv(CSV_PATH).to_dict('records')
           if CSV_PATH.exists() and (not RUN_FITS or RESUME_FITS) else [])
print('Device:', 'cuda' if DEVICE == 'auto' and torch.cuda.is_available()
      else 'cpu' if DEVICE == 'auto' else DEVICE)
print(f'{len(records)} existing CSV records; evaluation seeds={SEEDS}; updates={N_UPDATES}.')
print(f'Calibration seeds={CALIBRATION_SEEDS}; candidates={LAMBDA_CANDIDATES}; '
      f'calibration updates={N_CALIBRATION_UPDATES}.')

def normalize_rows(r):
    r = np.asarray(r, np.float64); return r / np.linalg.norm(r, axis=1, keepdims=True)

def simulate(n_train, n_test, p, k, sparsity, snr_db, data_seed):
    seeds = np.random.SeedSequence(data_seed).spawn(3)
    dict_rng, train_rng, test_rng = (np.random.default_rng(s) for s in seeds)
    D = normalize_rows(dict_rng.uniform(-1, 1, (k, p)))
    def codes(rng, n):
        out = np.zeros((n, k))
        for i in range(n):
            sup = rng.choice(k, sparsity, replace=False); out[i, sup] = rng.uniform(0, 1, sparsity)
        return out
    def add_noise(rng, clean):
        raw = rng.standard_normal(clean.shape)
        return clean + raw * (np.linalg.norm(clean) * 10 ** (-snr_db / 20) / np.linalg.norm(raw))
    a_tr, a_te = codes(train_rng, n_train), codes(test_rng, n_test)
    return dict(x_train=add_noise(train_rng, a_tr @ D).astype(np.float32),
                x_test=add_noise(test_rng, a_te @ D).astype(np.float32), dictionary=D)


def starting_atoms(x, k, fit_seed):
    rng = np.random.default_rng(fit_seed)
    indices = rng.choice(x.shape[0], k, replace=False)
    return normalize_rows(x[indices]).astype(np.float32)


def make_weight_rule(method):
    if method not in METHODS:
        raise ValueError(f'Unknown method: {method}')
    state = {'last_cap': np.nan}

    def weight_fn(m):
        if method == 'ordinary':
            return m.clamp_min(0)
        if method == 'active_sigmoid':
            return (m > 0) * torch.sigmoid(m / SIGMOID_T)
        if method == 'sigmoid':
            return torch.sigmoid(m / SIGMOID_T)
        positive = m[m > 0]
        cap = torch.quantile(positive, CAP_QUANTILE) if positive.numel() else m.new_zeros(())
        state['last_cap'] = cap
        hard = m.clamp_min(0).clamp_max(cap)
        if method == 'clipped':
            return hard
        softplus = torch.nn.functional.softplus
        return hard + SMOOTH_T * (
            softplus(-m.abs() / SMOOTH_T) - softplus(-(m - cap).abs() / SMOOTH_T))

    return weight_fn, state


def recovery_metrics(atoms, true_atoms):
    cosine = normalize_rows(true_atoms) @ normalize_rows(atoms).T
    true_ids, learned_ids = linear_sum_assignment(-cosine)
    matched = cosine[true_ids, learned_ids]
    cover = cosine.max(axis=1)
    return dict(mean_cosine=float(matched.mean()),
                matched60=int((matched >= 0.6).sum()), matched90=int((matched >= 0.9).sum()),
                cover60=int((cover >= 0.6).sum()), cover90=int((cover >= 0.9).sum()))


def heldout_metrics(model, x_test, alpha):
    codes = model.transform(x_test)
    residual = np.asarray(x_test, np.float64) - np.asarray(codes, np.float64) @ np.asarray(
        model.components_, np.float64)
    objective = (0.5 * np.square(residual).sum() + alpha * codes.astype(np.float64).sum()) / len(x_test)
    diagnostics = {f'test_{key}': value for key, value in model.transform_diagnostics_.items()}
    return dict(test_objective=float(objective), **diagnostics)


@torch.no_grad()
def final_margin_diagnostics(model, x_train, alpha):
    previous_precision = torch.get_float32_matmul_precision()
    torch.set_float32_matmul_precision('highest')
    try:
        device = model.device_
        x = torch.as_tensor(x_train, dtype=torch.float32, device=device)
        a = torch.as_tensor(model.coefficients_, dtype=torch.float32, device=device)
        s = torch.as_tensor(model.components_, dtype=torch.float32, device=device)
        residual = x - a @ s
        margins = a + residual @ s.T - alpha
        codes_active, margins_active = a > 0, margins > 0
        tolerance = CODE_TOL * a.abs().amax(1).clamp_min(1)[:, None]
        sigmoid = torch.sigmoid(margins / SIGMOID_T)
        disagreement = (codes_active != margins_active) * sigmoid
        l1_difference = disagreement.double().sum()
        active_weight_mass = (margins_active * sigmoid).double().sum()
        norms = s.norm(dim=1)
        unit = s / norms[:, None]
        atom_cosine = unit @ unit.T
        off_diagonal_mean = (atom_cosine.double().sum() - atom_cosine.diag().double().sum()) / (len(s) * (len(s) - 1))
        return dict(n_margin_entries=margins.numel(), n_positive_codes=int(codes_active.sum().item()),
                    n_positive_margins=int(margins_active.sum().item()),
                    zero_codes_positive_margin=int(((~codes_active) & margins_active).sum().item()),
                    near_zero_margin_count=int((margins.abs() <= tolerance).sum().item()),
                    active_mask_disagreement_count=int((codes_active != margins_active).sum().item()),
                    active_mask_weight_l1_difference=float(l1_difference.item()),
                    active_mask_weight_relative_l1_difference=float((l1_difference / active_weight_mass.clamp_min(1e-12)).item()),
                    norm_mean_atom=float(unit.double().mean(0).norm().item()),
                    mean_offdiag_atom_cosine=float(off_diagonal_mean.item()),
                    residual_mean_norm=float(residual.double().mean(0).norm().item()),
                    mean_residual_row_norm=float(residual.double().norm(dim=1).mean().item()))
    finally:
        torch.set_float32_matmul_precision(previous_precision)

def settings():
    # The identity changes when a relevant experiment setting or shared model changes.
    return dict(protocol='paired_margin_evidence_v1', n_train=N_TRAIN, n_test=N_TEST,
                p=P, k=K, sparsity=SPARSITY, snr_db=SNR, evaluation_seeds=list(SEEDS),
                calibration_seeds=list(CALIBRATION_SEEDS), lambda_candidates=list(LAMBDA_CANDIDATES),
                lambda_step=LAMBDA_STEP, lambda_extension='improving_boundary_positive_v1',
                n_evaluation_updates=N_UPDATES, n_calibration_updates=N_CALIBRATION_UPDATES,
                lr=LEARNING_RATE, code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER,
                record_every=RECORD_EVERY, cap_quantile=CAP_QUANTILE, sigmoid_t=SIGMOID_T,
                smooth_t=SMOOTH_T, initial_codes='zero_then_certified_refit',
                device_requested=DEVICE, cpu_threads=2, model_sha256=MODEL_SHA256,
                numpy_version=np.__version__, torch_version=torch.__version__)


def current_settings_id():
    if not RUN_FITS:
        return PLOT_SETTINGS_ID
    return hashlib.sha256(json.dumps(settings(), sort_keys=True).encode()).hexdigest()[:16]


def save_records():
    if not RUN_FITS:
        raise RuntimeError('Plot-only mode does not modify the score CSV.')
    temporary_path = CSV_PATH.with_suffix('.csv.tmp')
    pd.DataFrame.from_records(records).to_csv(temporary_path, index=False)
    temporary_path.replace(CSV_PATH)


def fit_key(stage, seed, method, alpha, n_updates):
    key = [current_settings_id(), stage, int(seed), method, float(alpha), int(n_updates)]
    return hashlib.sha256(json.dumps(key).encode()).hexdigest()[:20]


def completed_fit(stage, seed, method, alpha, n_updates):
    key = fit_key(stage, seed, method, alpha, n_updates)
    matches = [r for r in records if r.get('record_type') == 'fit'
               and r.get('status') == 'complete' and r.get('run_key') == key]
    if len(matches) > 1:
        raise RuntimeError(f'Duplicate completed rows for {key}; inspect {CSV_PATH}.')
    return matches[0] if matches else None


def run_fit(stage, seed, method, alpha, n_updates=None, verbose=True):
    if not RUN_FITS:
        raise RuntimeError('Set RUN_FITS=True to run model fits.')
    if n_updates is None:
        n_updates = N_CALIBRATION_UPDATES if stage == 'calibration' else N_UPDATES
    previous = completed_fit(stage, seed, method, alpha, n_updates)
    if previous is not None:
        print(f'Reuse {stage}: seed={seed}, {method}, lambda={alpha:g}, '
              f'cosine={previous["mean_cosine"]:.5f}', flush=True)
        return previous
    data = simulate(N_TRAIN, N_TEST, P, K, SPARSITY, SNR, seed)
    initial_atoms = starting_atoms(data['x_train'], K, seed)
    weight_fn, weight_state = make_weight_rule(method)
    run_key = fit_key(stage, seed, method, alpha, n_updates)
    metadata = dict(settings_id=current_settings_id(), run_key=run_key, stage=stage,
                    seed=int(seed), method=method, alpha=float(alpha), n_updates=int(n_updates),
                    **settings())
    start = time.perf_counter()

    def progress(iteration, dictionary):
        scores = recovery_metrics(dictionary, data['dictionary'])
        elapsed = time.perf_counter() - start
        eta = elapsed * (n_updates - iteration) / iteration if iteration else np.nan
        cap = weight_state['last_cap']
        cap = float(cap.item()) if isinstance(cap, torch.Tensor) else float(cap)
        # The model's bar adds current coding gap, inner steps, movement, and elapsed time.
        return dict(cosine=scores['mean_cosine'], matched60=scores['matched60'],
                    matched90=scores['matched90'], update_cap=cap, eta_seconds=eta)

    model = MarginDictionaryLearning(
        K, weight_fn=weight_fn, alpha=alpha, lr=LEARNING_RATE, n_iter=n_updates,
        code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER,
        dict_init=initial_atoms, code_init=np.zeros((N_TRAIN, K), dtype=np.float32),
        random_state=seed, device=DEVICE, record_every=RECORD_EVERY,
        verbose=verbose, callback=progress)
    print(f'Start {stage}: seed={seed}, {method}, lambda={alpha:g}, updates={n_updates}.', flush=True)
    try:
        model.fit(data['x_train'])
        scores = recovery_metrics(model.components_, data['dictionary'])
        scores.update(heldout_metrics(model, data['x_test'], alpha))
        scores.update(final_margin_diagnostics(model, data['x_train'], alpha))
        last = model.history_[-1]
        tail = [h for h in model.history_ if h['iteration'] >= max(0, n_updates - 1000)]
        scores.update(last1000_cosine_change=tail[-1]['cosine'] - tail[0]['cosine'],
                      last1000_cosine_range=max(h['cosine'] for h in tail) - min(h['cosine'] for h in tail),
                      last1000_objective_change=tail[-1]['objective'] - tail[0]['objective'])
        row = dict(metadata, record_type='fit', status='complete', iteration=model.n_iter_,
                   seconds=time.perf_counter() - start, training_seconds=model.training_time_,
                   device_used=model.device_, objective=last['objective'],
                   max_atom_change=last['max_atom_change'], mean_inner_steps=last['mean_inner_steps'],
                   final_cap=last['update_cap'], **scores,
                   **{f'train_{key}': value for key, value in model.final_coding_diagnostics_.items()})
    except Exception as error:
        # Preserve partial certified history and a visible failure, then stop the cell.
        records.extend(dict(metadata, record_type='trace', status='interrupted', **h)
                       for h in getattr(model, 'history_', []))
        records.append(dict(metadata, record_type='failure', status='failed',
                            seconds=time.perf_counter() - start,
                            error=f'{type(error).__name__}: {error}'))
        save_records()
        print(f'FAILED {stage}: seed={seed}, {method}, lambda={alpha:g}: {error}', flush=True)
        raise
    # On a successful retry, replace any partial traces while retaining recorded failures.
    records[:] = [r for r in records if not (r.get('run_key') == run_key and r.get('record_type') == 'trace')]
    records.extend(dict(metadata, record_type='trace', status='complete', **h) for h in model.history_)
    records.append(row)
    save_records()
    print(f'Complete seed={seed} {method}: cosine={row["mean_cosine"]:.5f}, '
          f'matched .6/.9={row["matched60"]}/{row["matched90"]}, '
          f'test objective={row["test_objective"]:.5f}, '
          f'train/test gaps={row["train_relative_gap"]:.2e}/{row["test_relative_gap"]:.2e}, '
          f'elapsed={row["seconds"]:.0f}s.', flush=True)
    return row


def frozen_selection():
    matches = [r for r in records if r.get('record_type') == 'selection'
               and r.get('status') == 'complete' and r.get('settings_id') == current_settings_id()]
    if len(matches) > 1:
        raise RuntimeError('Multiple frozen selections for the same settings; inspect the CSV.')
    return matches[0] if matches else None


def run_calibration(verbose=True):
    selected = frozen_selection()
    if selected is not None:
        print(f'Reuse frozen lambda={selected["alpha"]:g}.', flush=True)
        return float(selected['alpha'])
    if set(CALIBRATION_SEEDS) & set(SEEDS):
        raise ValueError('Calibration and evaluation seeds must be disjoint.')
    candidates = sorted(set(LAMBDA_CANDIDATES))
    if LAMBDA_STEP <= 0 or not candidates or min(candidates) <= 0:
        raise ValueError('Use positive penalty candidates and a positive extension step.')
    rows, previous_best = [], None
    pending = candidates.copy()
    while True:
        rows += [run_fit('calibration', seed, 'ordinary', alpha, N_CALIBRATION_UPDATES, verbose)
                 for alpha in pending for seed in CALIBRATION_SEEDS]
        frame = pd.DataFrame(rows)
        if (frame.duplicated(['alpha', 'seed']).any()
                or not np.isfinite(frame['mean_cosine']).all()):
            raise RuntimeError('Calibration requires unique, finite results for each candidate and seed.')
        table = frame.groupby('alpha')['mean_cosine'].agg(['mean', 'std', 'count'])
        if not (table['count'] == len(CALIBRATION_SEEDS)).all():
            raise RuntimeError('Every calibration candidate must have all independent seeds.')
        # Ties favour the smaller penalty; evaluation results never enter selection.
        chosen = min(candidates, key=lambda value: (-table.loc[value, 'mean'], value))
        best = float(table.loc[chosen, 'mean'])
        improved = previous_best is None or best > previous_best
        pending = []
        # Same rule as the other notebooks: try an edge once, then require improvement.
        if improved:
            lower = round(min(candidates) - LAMBDA_STEP, 10)
            if chosen == min(candidates) and lower > 0:
                pending.append(lower)
            if chosen == max(candidates):
                pending.append(round(max(candidates) + LAMBDA_STEP, 10))
        if not pending:
            break
        print(f'Extending ordinary calibration to lambda={pending}; '
              f'current best={chosen:g}, mean cosine={best:.5f}.', flush=True)
        candidates = sorted(set(candidates + pending))
        previous_best = best
    row = dict(record_type='selection', status='complete', stage='calibration',
               settings_id=current_settings_id(), alpha=float(chosen),
               selection_metric='mean final signed Hungarian cosine on ordinary calibration fits',
               selection_tie_break='smaller alpha', searched_lambdas=candidates, **settings())
    for alpha in candidates:
        row[f'calibration_mean_alpha_{alpha:g}'] = float(table.loc[alpha, 'mean'])
    records.append(row)
    save_records()  # Freeze before evaluation can begin.
    display(table)
    print(f'Frozen lambda={chosen:g}; selection saved to {CSV_PATH}.', flush=True)
    return float(chosen)


def run_evaluation(verbose=True):
    selected = frozen_selection()
    if selected is None:
        raise RuntimeError('Run the calibration cell first to freeze lambda before evaluation.')
    alpha = float(selected['alpha'])
    return [run_fit('evaluation', seed, method, alpha, N_UPDATES, verbose)
            for seed in SEEDS for method in METHODS]

if RUN_FITS:
    if records and {row.get('settings_id') for row in records} != {current_settings_id()}:
        raise ValueError('Cannot resume this checkpoint with the current code/settings. '
                         'Set RESUME_FITS=False for a fresh run, or RUN_FITS=False to redraw saved results.')
    LAMBDA = run_calibration()
    evaluation_rows = run_evaluation()
else:
    # Plot recorded fits using their original identity, independently of the local
    # NumPy/Torch version or device. Fit settings must still match; search settings
    # and seed counts describe a future run, so plotting uses the saved seed list.
    plot_settings = {key: value for key, value in settings().items()
                     if key not in ('numpy_version', 'torch_version', 'device_requested', 'model_sha256',
                                    'evaluation_seeds', 'calibration_seeds', 'lambda_candidates',
                                    'lambda_step', 'lambda_extension')}

    def matches_plot_settings(row):
        for key, value in plot_settings.items():
            stored = row.get(key)
            if isinstance(value, list) and isinstance(stored, str):
                stored = json.loads(stored)
            if stored != value:
                return False
        return True

    selections = [row for row in records if row.get('record_type') == 'selection'
                  and row.get('status') == 'complete' and matches_plot_settings(row)]
    if len(selections) != 1:
        raise RuntimeError('Expected one saved calibration matching these settings in evidence_scores.csv; '
                           'set RUN_FITS=True to generate results when no matching records exist.')
    saved_seeds = selections[0]['evaluation_seeds']
    SEEDS = tuple(json.loads(saved_seeds) if isinstance(saved_seeds, str) else saved_seeds)
    PLOT_SETTINGS_ID = selections[0]['settings_id']
    LAMBDA = float(selections[0]['alpha'])
    evaluation_rows = [completed_fit('evaluation', seed, method, LAMBDA, N_UPDATES)
                       for seed in SEEDS for method in METHODS]
    print(f'Plot-only mode: {len(SEEDS)} saved evaluation seeds, settings {PLOT_SETTINGS_ID}, '
          f'lambda={LAMBDA:g}; no fitting or CSV writes.')

def mean_sd(values, digits=4, signed=False):
    values = np.asarray(values, dtype=float)
    prefix = '+' if signed else ''
    return f'{values.mean():{prefix}.{digits}f} Â± {values.std(ddof=1):.{digits}f}'


def export_evidence_figures(frame):
    output = OUTPUT_DIR
    with plt.rc_context():
        plt.rcdefaults()
        methods = list(METHODS)
        labels = ['Ordinary', 'Clipped', 'Active\nsigmoid', 'Full\nsigmoid', 'Smooth\nclipped']
        colors = {'ordinary':'#657383', 'clipped':'#2678AC', 'active_sigmoid':'#BC8B29',
                  'sigmoid':'#CD7356', 'smooth_clipped':'#258C73'}
        assert len(frame) == len(SEEDS) * len(METHODS) and not frame.duplicated(['seed', 'method']).any()
        assert set(frame.method) == set(methods) and set(frame.seed) == set(SEEDS)
        assert frame.alpha.nunique() == 1 and frame.alpha.iloc[0] == LAMBDA
        assert frame.cap_quantile.eq(CAP_QUANTILE).all() and frame.smooth_t.eq(SMOOTH_T).all()
        seeds = sorted(frame.seed.unique())
        positions = np.arange(len(methods))
        offsets = np.linspace(-.10, .10, len(seeds))
        plt.rcParams.update({
            'font.family': 'STIXGeneral', 'mathtext.fontset':'stix', 'font.size':10,
            'axes.labelsize':10, 'xtick.labelsize':9, 'ytick.labelsize':9,
            'axes.linewidth':.6, 'axes.edgecolor':'#8C939A',
            'xtick.color':'#39434A', 'ytick.color':'#39434A', 'text.color':'#263238',
            'axes.labelcolor':'#263238', 'axes.spines.top':False, 'axes.spines.right':False,
            'pdf.fonttype':42, 'ps.fonttype':42,
        })

        def style(ax, grid_axis):
            ax.set_axisbelow(True)
            ax.grid(axis=grid_axis, color='#E8EBEE', linewidth=.6)
            ax.tick_params(length=3, width=.6, pad=5)
            ax.spines['left'].set_color('#A4ABB1')
            ax.spines['bottom'].set_color('#A4ABB1')

        def save(fig, ax, name):
            assert ax.get_title() == '' and fig._suptitle is None and not fig.texts
            assert all(c not in ax.get_xlabel()+ax.get_ylabel() for c in ('\u2191','\u2193'))
            assert not any(line.get_marker() == 'D' for line in ax.lines)
            fig.tight_layout(pad=.7)
            fig.savefig(output / f'{name}.pdf', bbox_inches='tight', pad_inches=.05)
            print(output / f'{name}.pdf')
            display(fig)
            plt.close(fig)

        for metric, ylabel, name in [
            ('mean_cosine', 'Mean signed matched cosine', 'fig_evidence_recovery'),
            ('test_objective', 'Held-out objective per observation', 'fig_evidence_heldout'),
        ]:
            values = frame.pivot(index='seed', columns='method', values=metric).loc[seeds, methods]
            fig, ax = plt.subplots(figsize=(5.4, 3.25))
            style(ax, 'y')
            for offset, seed in zip(offsets, seeds):
                ax.plot(positions + offset, values.loc[seed].to_numpy(),
                        color='#B8C1C8', linewidth=.7, alpha=.52, zorder=1)
            for j, method in enumerate(methods):
                y = values[method].to_numpy()
                ax.scatter(j + offsets, y, s=29, color=colors[method],
                           edgecolors='white', linewidths=.55, alpha=.95, zorder=3)
            ax.set(xticks=positions, xticklabels=labels, ylabel=ylabel, xlim=(-.5, 4.5))
            ax.tick_params(axis='x', length=0)
            ax.yaxis.set_major_locator(MultipleLocator(.01) if metric == 'test_objective' else MaxNLocator(nbins=6))
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
            ax.margins(y=.12)
            save(fig, ax, name)

        cosine = frame.pivot(index='seed', columns='method', values='mean_cosine').loc[seeds, methods]
        contrasts = [('clipped','ordinary'), ('smooth_clipped','ordinary'),
                     ('smooth_clipped','clipped'), ('sigmoid','active_sigmoid')]
        contrast_labels = ['Clipped \u2212 Ordinary', 'Smooth clipped \u2212 Ordinary',
                           'Smooth clipped \u2212 Clipped', 'Full sigmoid \u2212 Active sigmoid']
        fig, ax = plt.subplots(figsize=(6.3, 2.9))
        style(ax, 'x')
        ax.axvline(0, color='#929BA3', linewidth=.8, linestyle=(0,(3,3)), zorder=1)
        for j, (method, reference) in enumerate(contrasts):
            difference = (cosine[method] - cosine[reference]).to_numpy()
            ax.scatter(difference, j + offsets, s=30, color=colors[method],
                       edgecolors='white', linewidths=.55, alpha=.95, zorder=3)
        ax.set(yticks=np.arange(len(contrasts)), yticklabels=contrast_labels,
               xlabel='Paired difference in matched cosine', ylim=(3.5,-.5))
        ax.tick_params(axis='y', length=0)
        ax.xaxis.set_major_locator(MultipleLocator(.02))
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.margins(x=.10)
        save(fig, ax, 'fig_evidence_contrasts')


def summarise_and_plot():
    selected = frozen_selection()
    if selected is None:
        raise RuntimeError('No frozen penalty selection is available.')
    alpha = float(selected['alpha'])
    expected = [(seed, method) for seed in SEEDS for method in METHODS]
    rows = [completed_fit('evaluation', seed, method, alpha, N_UPDATES) for seed, method in expected]
    missing = [pair for pair, row in zip(expected, rows) if row is None]
    if missing:
        raise RuntimeError(f'Evaluation is incomplete; missing completed fits: {missing}')
    frame = pd.DataFrame(rows)
    metrics = [('mean_cosine', 'Matched cosine', 4), ('matched60', 'Matched â‰¥0.6', 1),
               ('matched90', 'Matched â‰¥0.9', 1), ('cover60', 'Covered â‰¥0.6', 1),
               ('cover90', 'Covered â‰¥0.9', 1), ('test_objective', 'Held-out objective â†“', 4)]
    summary = pd.DataFrame([
        {'Method': LABELS[method], **{label: mean_sd(frame.loc[frame.method == method, key], digits)
                                    for key, label, digits in metrics}}
        for method in METHODS]).set_index('Method')
    print(f'Frozen lambda={alpha:g}; n={len(SEEDS)} paired seeds; mean Â± sample SD.')
    display(summary)
    cosine = frame.pivot(index='seed', columns='method', values='mean_cosine').loc[list(SEEDS), list(METHODS)]
    objective = frame.pivot(index='seed', columns='method', values='test_objective').loc[list(SEEDS), list(METHODS)]
    contrasts = [(method, 'ordinary') for method in METHODS if method != 'ordinary']
    contrasts += [('sigmoid', 'active_sigmoid'), ('smooth_clipped', 'clipped')]
    paired = []
    for method, reference in contrasts:
        delta = cosine[method] - cosine[reference]
        obj_delta = objective[method] - objective[reference]
        paired.append({'Contrast': f'{LABELS[method]} âˆ’ {LABELS[reference]}',
                       'Î” matched cosine': mean_sd(delta, signed=True),
                       'Cosine wins': f'{int((delta > 0).sum())}/{len(delta)}',
                       'Î” held-out objective': mean_sd(obj_delta, signed=True),
                       'Objective wins (lower)': f'{int((obj_delta < 0).sum())}/{len(obj_delta)}'})
    display(pd.DataFrame(paired).set_index('Contrast'))
    diagnostic_keys = ['train_relative_margin_error', 'train_relative_gap',
                       'test_relative_margin_error', 'test_relative_gap']
    print('Largest final coding certificate values:')
    print(frame[diagnostic_keys].max().to_string(float_format=lambda value: f'{value:.3e}'))
    if (frame[diagnostic_keys] > CODE_TOL).any().any():
        raise RuntimeError('A saved coding certificate exceeds the configured tolerance.')
    drift_rows = []
    for row in rows:
        traces = [r for r in records if r.get('record_type') == 'trace' and r.get('status') == 'complete'
                  and r.get('run_key') == row['run_key']]
        trace = pd.DataFrame(traces).sort_values('iteration')
        if len(trace) == 0 or int(trace.iloc[-1]['iteration']) != N_UPDATES:
            raise RuntimeError(f'Missing final trace for {row["run_key"]}.')
        before = trace[trace.iteration <= max(0, N_UPDATES - 500)].iloc[-1]
        after = trace.iloc[-1]
        drift_rows.append(dict(method=LABELS[row['method']], seed=row['seed'],
                               last500_cosine_change=after['cosine'] - before['cosine'],
                               last500_objective_change=after['objective'] - before['objective'],
                               final_max_atom_change=row['max_atom_change']))
    drift = pd.DataFrame(drift_rows)
    print('Finite-budget diagnostics (mean across seeds; full traces are in the CSV):')
    display(frame.groupby('method', sort=False)[['last1000_cosine_change', 'last1000_cosine_range',
                                                 'last1000_objective_change']].mean())
    display(drift.groupby('method', sort=False)[['last500_cosine_change',
                                               'last500_objective_change', 'final_max_atom_change']].mean())

    print('Final margin and dictionary diagnostics (mean across seeds):')
    display(frame.groupby('method', sort=False)[['n_positive_codes', 'n_positive_margins',
        'zero_codes_positive_margin', 'near_zero_margin_count', 'active_mask_weight_relative_l1_difference',
        'norm_mean_atom', 'mean_offdiag_atom_cosine', 'residual_mean_norm']].mean())

    export_evidence_figures(frame)
    return frame, summary, pd.DataFrame(paired), drift


evaluation_frame, summary_table, paired_table, drift_table = summarise_and_plot()
